# 测试 `relay.qnn.simulated_quantize`

In [1]:
import set_env

In [ ]:
import tvm
from tvm import te
import numpy as np
from tvm import relay
from tvm.contrib import graph_executor
from tvm.runtime.vm import VirtualMachine
from tvm.topi.nn.qnn import SQNN_DTYPE_TO_CODE
from utils import allclose_with_rounding, quantize_test_driver

In [ ]:
def allclose_with_rounding(a, b):
    """
    比较两个数组是否足够接近，允许一定数量的四舍五入误差。
    用于QNN模拟量化测试中，验证量化前后结果的一致性。

    参数:
    a: 第一个比较数组 (NumPy数组或TVM张量)
    b: 第二个比较数组 (NumPy数组或TVM张量)

    异常:
    AssertionError: 如果不匹配元素数量超过3个

    说明:
    - 函数通过逐元素比较创建布尔数组mismatch
    - 使用np.sum计算不匹配元素总数
    - 允许最多3个不匹配元素，以容忍GPU浮点运算的舍入误差
    """
    # 创建布尔数组，True表示对应位置元素不相等
    mismatch = a != b
    # 计算不匹配元素总数并断言不超过3个
    # 容忍少量误差是因为GPU的fp32算术可能有舍入误差
    assert np.sum(mismatch) <= 3

def quantize_test_driver(in_dtype, quant_args, axis, out_dtype, in_data):
    """
    QNN模拟量化测试的驱动函数，用于执行量化操作并返回结果。

    该函数构建一个包含QNN量化操作的Relay计算图，编译并执行它，
    最后返回量化后的结果。主要用于测试模拟量化功能的正确性。

    参数:
    in_dtype (str): 输入数据的数据类型，如 'float32'
    quant_args (dict): 量化参数字典，必须包含以下键:
        - 'out_zero_point': 输出量化的零点值
        - 'out_scale': 输出量化的缩放因子
    axis (int): 量化操作应用的轴
    out_dtype (str): 输出数据的数据类型，如 'int8', 'uint8'
    in_data (numpy.ndarray): 输入数据数组

    返回:
    numpy.ndarray: 量化后的输出数据
    """
    # 获取输入数据的形状
    shape = in_data.shape
    # 创建Relay变量表示输入数据
    input_data = relay.var("input_data", shape=shape, dtype=in_dtype)
    # 创建输出零点和缩放因子的常量
    output_zero_point = relay.const(quant_args["out_zero_point"])
    output_scale = relay.const(quant_args["out_scale"])
    # 执行QNN量化操作
    quantized_output = relay.qnn.quantize(
        input_data,
        output_scale=output_scale,
        output_zero_point=output_zero_point,
        axis=axis,
        out_dtype=out_dtype,
    )
    # 构建Relay函数和IRModule
    mod = relay.Function(relay.analysis.free_vars(quantized_output), quantized_output)
    mod = tvm.IRModule.from_expr(mod)
    # 编译模型

    dev = tvm.cpu(0)
    target = "llvm"
    with tvm.transform.PassContext(opt_level=3):
        lib = relay.build(mod, target=target)
    # 创建运行时模块
    rt_mod = graph_executor.GraphModule(lib["default"](dev))
    # 设置输入数据
    rt_mod.set_input(input_data=in_data)
    # 设置模型参数
    # rt_mod.set_input(**params)
    # 运行模型
    rt_mod.run()
    # 获取输出结果
    res = rt_mod.get_output(0).numpy()
    return res


In [5]:
def build_simulated_quantize(input_data, scale, zp, dtype, axis=-1):
    """
    构建用于QNN模拟量化的虚拟机执行环境。

    该函数创建一个包含QNN模拟量化操作的计算图，编译并优化它，
    然后返回一个可以执行此量化操作的虚拟机实例。主要用于测试
    模拟量化功能的性能和正确性。

    参数:
    input_data (relay.Expr): 输入的Relay表达式
    scale (relay.Expr): 量化缩放因子
    zp (relay.Expr): 量化零点
    dtype (str): 输出数据类型，如 'int8', 'uint8'
    axis (int, optional): 量化操作应用的轴，默认为-1（最后一个维度）

    返回:
    tvm.relay.vm.VirtualMachine: 可以执行模拟量化操作的虚拟机实例
    """
    # 创建QNN模拟量化操作
    sim_q = relay.qnn.simulated_quantize(
        input_data,
        scale,
        zp,
        axis=axis,
        out_dtype=dtype,
    )
    # 从表达式构建IR模块
    mod = tvm.IRModule.from_expr(sim_q)
    # 编译模块，设置优化级别为3
    with tvm.transform.PassContext(opt_level=3):
        vm_exec = relay.vm.compile(mod, "llvm", params=None)
    # 创建虚拟机实例
    vm = VirtualMachine(vm_exec, tvm.cpu(0))
    return vm


In [6]:
def verify_simulated_quantize_simple(dtype):
    """
    验证QNN模拟量化功能的简单测试函数。

    该函数通过生成随机测试数据，分别使用常规量化和模拟量化两种方式
    执行量化操作，然后比较两种方式的结果是否一致，以验证模拟量化
    功能的正确性。

    参数:
    dtype (str): 输出数据类型，如 'int8', 'uint8'

    步骤:
    1. 生成随机测试数据
    2. 设置量化参数（缩放因子、零点）
    3. 使用quantize_test_driver执行常规量化
    4. 创建Relay变量和计算图
    5. 构建模拟量化虚拟机
    6. 执行模拟量化
    7. 比较两种量化结果是否一致
    """
    # 生成随机测试数据，范围[-128, 127)，形状[2, 5]，类型float32
    data = np.random.uniform(low=-128, high=127, size=[2, 5]).astype("float32")
    # 设置量化缩放因子为0.5
    scale_np = np.float32(0.5)
    # 设置量化零点为127
    zp_np = np.int32(127)
    # 将数据类型字符串转换为对应的整数编码
    dtype_np = np.int32(SQNN_DTYPE_TO_CODE[dtype])
    # 准备量化参数字典
    quant_args = {"out_zero_point": zp_np, "out_scale": scale_np}
    # 使用常规量化方法得到预期输出
    q_out = quantize_test_driver(
        in_dtype="float32",
        quant_args=quant_args,
        axis=-1,
        out_dtype=dtype,
        in_data=data,
    )
    # 创建Relay变量表示输入数据
    input_data = relay.var("input_data", shape=data.shape, dtype="float32")
    # 创建Relay变量表示缩放因子
    scale = relay.var("scale", shape=[])
    # 创建Relay变量表示零点
    zp = relay.var("zp", shape=[], dtype="int32")
    # 创建Relay变量表示数据类型
    dtype = relay.var("dtype", shape=[], dtype="int32")
    # 构建模拟量化虚拟机
    vm = build_simulated_quantize(input_data, scale, zp, dtype)
    # 在虚拟机上执行模拟量化
    sim_q_out = vm.invoke("main", input_data=data, scale=scale_np, zp=zp_np, dtype=dtype_np)
    # 比较模拟量化结果与预期输出是否一致（容忍浮点误差）
    allclose_with_rounding(sim_q_out.numpy(), q_out)



## QNN 模拟量化功能测试

In [7]:
def test_simulated_quantize():
    """
    QNN模拟量化功能的主测试函数。

    该函数通过调用verify_simulated_quantize_simple函数，对三种不同的
    输出数据类型（无符号8位整数、有符号8位整数和有符号32位整数）
    进行模拟量化测试，以验证模拟量化功能在不同数据类型下的正确性。
    """
    # 测试无符号8位整数（uint8）的模拟量化
    verify_simulated_quantize_simple("uint8")
    # 测试有符号8位整数（int8）的模拟量化
    verify_simulated_quantize_simple("int8")
    # 测试有符号32位整数（int32）的模拟量化
    verify_simulated_quantize_simple("int32")

In [8]:
test_simulated_quantize()

## 测试动态通道的QNN模拟量化功能

In [9]:
def test_dynamic_channels():
    """
    测试动态通道的QNN模拟量化功能。
    
    该函数验证模拟量化能否在不重新编译的情况下，同时支持标量量化参数和每通道量化参数。
    主要测试流程：
    1. 生成随机测试数据
    2. 使用标量量化参数进行测试
    3. 创建具有未定义形状的变量并构建模拟量化虚拟机
    4. 使用标量参数运行模拟量化并验证结果
    5. 使用每通道量化参数进行测试（无需重新编译虚拟机）
    6. 验证每通道量化结果
    
    此测试重点在于验证模拟量化对动态通道配置的支持能力。
    """
    # 编译一次模拟量化，但支持每通道或标量参数
    data = np.random.uniform(low=-64, high=64, size=[2, 5]).astype("float32")
    
    # 测试标量QNN参数
    scale_np = np.asarray([0.5]).astype("float32")  # 标量缩放因子
    zp_np = np.asarray([127]).astype("int32")       # 标量零点
    dtype_np = np.int32(SQNN_DTYPE_TO_CODE["uint8"])  # 输出数据类型代码
    quant_args = {"out_zero_point": zp_np[0], "out_scale": scale_np[0]}
    
    # 使用测试驱动函数获取量化输出
    q_out = quantize_test_driver(
        in_dtype="float32",    # 输入数据类型
        quant_args=quant_args, # 量化参数
        axis=0,                # 量化轴
        out_dtype="uint8",     # 输出数据类型
        in_data=data,          # 输入数据
    )
    
    # 创建具有未定义形状的变量，以支持动态参数
    input_data = relay.var("input_data", shape=data.shape, dtype="float32")
    scale = relay.var("scale", shape=[relay.Any()], dtype="float32")  # 使用relay.Any()允许任意形状
    zp = relay.var("zp", shape=[relay.Any()], dtype="int32")
    dtype = relay.var("dtype", shape=[], dtype="int32")
    
    # 构建模拟量化虚拟机
    vm = build_simulated_quantize(input_data, scale, zp, dtype, axis=0)
    
    # 使用标量输入运行模拟量化
    sim_q_out = vm.invoke("main", input_data=data, scale=scale_np, zp=zp_np, dtype=dtype_np)
    
    # 验证模拟量化结果与预期一致
    allclose_with_rounding(sim_q_out.numpy(), q_out)

    # 测试每通道量化参数，无需重新编译虚拟机
    scale_np = np.array([0.5, 0.25]).astype("float32")  # 每通道缩放因子
    zp_np = np.array([127, 123]).astype("int32")        # 每通道零点

    # 获取参考量化输出
    quant_args = {"out_zero_point": zp_np, "out_scale": scale_np}
    q_out = quantize_test_driver(
        in_dtype="float32",
        quant_args=quant_args,
        axis=0,
        out_dtype="uint8",
        in_data=data,
    )
    
    # 在不重新编译的情况下运行模拟量化并验证结果
    sim_q_out = vm.invoke("main", input_data=data, scale=scale_np, zp=zp_np, dtype=dtype_np)
    allclose_with_rounding(sim_q_out.numpy(), q_out)


In [10]:
test_dynamic_channels()

## 测试动态数据类型的QNN模拟量化功能

In [11]:
def test_dynamic_dtype():
    """
    测试动态数据类型的QNN模拟量化功能。
    
    该函数验证模拟量化能否在不重新编译的情况下，支持不同的输出数据类型量化。
    主要测试流程：
    1. 生成随机测试数据
    2. 测试float32到uint8的标量量化
    3. 创建具有未定义形状的变量并构建模拟量化虚拟机
    4. 使用标量参数运行模拟量化并验证结果
    5. 测试float32到int32的量化（无需重新编译虚拟机）
    6. 验证int32量化结果
    
    此测试重点在于验证模拟量化对动态数据类型配置的支持能力。
    """
    # 编译一次模拟量化，但支持不同类型的量化
    data = np.random.uniform(low=-64, high=64, size=[2, 5]).astype("float32")
    
    # 测试float32到uint8的标量量化
    scale_np = np.asarray([0.5]).astype("float32")  # 标量缩放因子
    zp_np = np.asarray([127]).astype("int32")       # 标量零点
    dtype_np = np.int32(SQNN_DTYPE_TO_CODE["uint8"])  # uint8类型代码
    quant_args = {"out_zero_point": zp_np[0], "out_scale": scale_np[0]}
    
    # 使用测试驱动函数获取uint8量化输出
    q_out = quantize_test_driver(
        in_dtype="float32",    # 输入数据类型
        quant_args=quant_args, # 量化参数
        axis=-1,               # 量化轴（最后一维）
        out_dtype="uint8",     # 输出数据类型
        in_data=data,          # 输入数据
    )
    
    # 创建具有未定义形状的变量，以支持动态参数
    input_data = relay.var("input_data", shape=data.shape, dtype="float32")
    scale = relay.var("scale", shape=[relay.Any()], dtype="float32")  # 任意形状的缩放因子
    zp = relay.var("zp", shape=[relay.Any()], dtype="int32")         # 任意形状的零点
    dtype = relay.var("dtype", shape=[], dtype="int32")              # 数据类型变量
    
    # 构建模拟量化虚拟机
    vm = build_simulated_quantize(input_data, scale, zp, dtype)
    
    # 使用标量输入运行模拟量化（uint8）
    sim_q_out = vm.invoke("main", input_data=data, scale=scale_np, zp=zp_np, dtype=dtype_np)
    
    # 验证模拟量化结果与预期一致
    allclose_with_rounding(sim_q_out.numpy(), q_out)

    # 测试float32到int32的量化，无需重新编译虚拟机
    
    # 获取int32参考量化输出
    q_out = quantize_test_driver(
        in_dtype="float32",
        quant_args=quant_args,
        axis=-1,
        out_dtype="int32",
        in_data=data,
    )
    
    # 更新数据类型为int32
    dtype_np = np.int32(SQNN_DTYPE_TO_CODE["int32"])
    
    # 在不重新编译的情况下运行模拟量化（int32）并验证结果
    sim_q_out = vm.invoke("main", input_data=data, scale=scale_np, zp=zp_np, dtype=dtype_np)
    allclose_with_rounding(sim_q_out.numpy(), q_out)


In [12]:
test_dynamic_dtype()